<a href="https://colab.research.google.com/github/AakashMahadik23/AI-Engineering/blob/main/SyntheticDataGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Synthetic Data Generator

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.5 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import torch
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
def load_model(model_name):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=quant_config
    )

    return tokenizer, model

In [ ]:
def build_prompt(dataset_type, num_samples):

    if dataset_type == "Customer Reviews":
        schema = """
Generate synthetic customer reviews in JSON format.
Each record must contain:
- review_text
- sentiment (positive, negative, neutral)
- product_category
"""

    elif dataset_type == "Bank Transactions":
        schema = """
Generate synthetic banking transactions in JSON format.
Each record must contain:
- transaction_id
- amount
- merchant
- transaction_type (debit/credit)
- fraud_label (yes/no)
"""

    prompt = f"""
{schema}

Generate {num_samples} diverse records.
Return ONLY a valid JSON array.
"""

    return prompt

In [ ]:
def generate_data(model_name, dataset_type, num_samples, temperature):

    tokenizer, model = load_model(model_name)

    prompt = build_prompt(dataset_type, num_samples)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1500,
        temperature=temperature,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Try extracting JSON
    try:
        json_start = response.find("[")
        json_end = response.rfind("]") + 1
        json_data = response[json_start:json_end]
        data = json.loads(json_data)
        df = pd.DataFrame(data)
    except:
        df = pd.DataFrame({"error": ["Model output could not be parsed. Try again."]})

    csv_path = "/content/synthetic_data.csv"
    df.to_csv(csv_path, index=False)

    return df, csv_path

In [ ]:
demo = gr.Interface(
    fn=generate_data,
    inputs=[
        gr.Dropdown(
            choices=[
                "mistralai/Mistral-7B-Instruct-v0.2",
                "Qwen/Qwen2.5-3B-Instruct"
            ],
            label="Select Model"
        ),
        gr.Dropdown(
            choices=[
                "Customer Reviews",
                "Bank Transactions"
            ],
            label="Dataset Type"
        ),
        gr.Slider(5, 50, value=10, step=5, label="Number of Samples"),
        gr.Slider(0.1, 1.5, value=0.7, step=0.1, label="Creativity (Temperature)")
    ],
    outputs=[
        gr.Dataframe(label="Generated Dataset Preview"),
        gr.File(label="Download CSV")
    ],
    title="Synthetic Data Generator",
    description="Generate structured synthetic datasets using open-source LLMs."
)

In [ ]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://da38185eec08f801fa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
